# 🔐 Notebook 1: Storing Passwords — Bad, Better, Best

**The big question:** *"Where do I put the user's password?"*

Whenever someone signs up to your app, you need to remember **something** so that next time they log in you can confirm it is really them. The most obvious idea is to just save the password somewhere — but that turns out to be a terrible idea. In this notebook we walk through three approaches:

1. 🟥 **BAD** — store the password in plain text.
2. 🟨 **BETTER** — store a SHA-256 hash of the password.
3. 🟩 **BEST** — store a **bcrypt** hash with a per-user salt and a tunable cost factor.

## Learning objectives
- Understand why hashing is necessary (database leaks happen).
- See why a *fast* hash like SHA-256 is still bad for passwords.
- Use `bcrypt` correctly and learn what the "cost factor" buys you.

## 🛠️ Setup

```bash
cd 01-foundations/authentication-authorization
uv sync
```

Then select the `.venv` kernel in VS Code (top-right of the notebook). If it does not appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 Approach 1 (BAD): plain text

We just store the password as the user typed it. If the database is ever leaked, every user's password is leaked too — and most people reuse passwords across sites.

In [ ]:
# A toy "users table" — just a Python dict
users_plain = {}

def signup_plain(username, password):
    users_plain[username] = password

def login_plain(username, password):
    return users_plain.get(username) == password

signup_plain("alice", "hunter2")
print("login ok?", login_plain("alice", "hunter2"))
print("login bad?", login_plain("alice", "wrong"))
print("DB contents:", users_plain)  # 😱 the password is right there

## 🟨 Approach 2 (BETTER): SHA-256 hash

A **hash function** takes any input and produces a fixed-size fingerprint. It is *one-way*: given the hash you cannot easily get the password back. So we store the hash, not the password.

This is better — but still wrong for passwords. SHA-256 is **fast**, which means an attacker with a leaked database can try **billions of guesses per second** on a GPU. Worse, two users with the same password get the same hash, which makes "rainbow tables" easy.

In [ ]:
import hashlib

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

users_sha = {}

def signup_sha(username, password):
    users_sha[username] = sha256_hex(password)

def login_sha(username, password):
    return users_sha.get(username) == sha256_hex(password)

signup_sha("alice", "hunter2")
signup_sha("bob", "hunter2")  # same password as alice
print("login ok?", login_sha("alice", "hunter2"))
print("DB:", users_sha)
# Notice: alice and bob have IDENTICAL hashes — easy to spot password reuse.

## 🟩 Approach 3 (BEST): bcrypt with salt + cost factor

`bcrypt` was designed for passwords. It does three important things:

1. **Salt** — a random value mixed into the hash so two users with the same password get different hashes.
2. **Slow on purpose** — it loops thousands of times. A "cost factor" of 12 means roughly `2^12 = 4096` rounds.
3. **Tunable** — as hardware gets faster, you bump the cost factor up.

Slow is good here: a legitimate login takes ~100 ms, but an attacker trying billions of guesses now takes years.

In [ ]:
import bcrypt
import time

users_bcrypt = {}

def signup_bcrypt(username, password, cost=12):
    # bcrypt generates a random salt for us and embeds it in the output hash
    hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt(rounds=cost))
    users_bcrypt[username] = hashed

def login_bcrypt(username, password):
    stored = users_bcrypt.get(username)
    if stored is None:
        return False
    return bcrypt.checkpw(password.encode(), stored)

signup_bcrypt("alice", "hunter2")
signup_bcrypt("bob", "hunter2")  # same password — different hash
for u, h in users_bcrypt.items():
    print(u, "->", h.decode())

In [ ]:
# Time a single login for different cost factors. Higher = slower = more secure.
for cost in (4, 8, 12):
    h = bcrypt.hashpw(b"hunter2", bcrypt.gensalt(rounds=cost))
    t0 = time.perf_counter()
    bcrypt.checkpw(b"hunter2", h)
    dt = (time.perf_counter() - t0) * 1000
    print(f"cost={cost:2d}  verify={dt:7.2f} ms")

## ✅ Recap

| Approach | Salt? | Slow? | Verdict |
|---|---|---|---|
| Plain text | – | – | Never. |
| SHA-256 | No | Very fast | No — easy to crack. |
| **bcrypt** | **Yes** | **Tunable** | **Use this** (or `argon2`, `scrypt`). |

Rule of thumb: **never write your own password storage**. Use a battle-tested library.